# List the tables in the database to verify

In [8]:
import pandas as pd
import numpy as np
import sqlite3
import os
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行
# set current working directory to the location of the database file
os.chdir('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/github')

# ---------------------------------------
# 宏观因子 -- Download from Michael W. McCracken 
# ---------------------------------------

In [9]:
# ==========================================================
# 开始日期晚于 1999-08-31 的月份
# ==========================================================
start_date = '1999-12-01'
end_date = '2024-12-31'

target_timeline = pd.date_range(start=start_date, end=end_date, freq='ME')
macro_config = pd.read_excel('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/MacroFactor.xlsx', sheet_name='Codex', index_col=0)
macro_cols = macro_config.index.dropna().unique().tolist()
# 构建承载最终结果的大面板 (初始化全为 NaN)
master_panel = pd.DataFrame(index=target_timeline, columns=macro_cols)
master_panel.index.name = 'mthcaldt'

print(f">>> Target Timeline: {start_date} to {end_date} ({len(target_timeline)} months)")
vintages_dir='/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/Historical FRED-MD Vintages Final'
output_save_path='/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data'

for cutoff_date in target_timeline:
    filename = cutoff_date.strftime('%Y-%m') + ".csv"
    file_path = os.path.join(vintages_dir, filename)
    df_raw_v = pd.read_csv(file_path)

    transform_dict = df_raw_v.iloc[0].to_dict() # 转换指令
    df_vintage = df_raw_v.iloc[1:].copy()        # 真实数据
    
    df_vintage['sasdate'] = pd.to_datetime(df_vintage['sasdate']) + pd.offsets.MonthEnd(0)  # 确保日期是月末
    df_vintage.set_index('sasdate', inplace=True)
    
    # 强制转成浮点数
    df_vintage = df_vintage.apply(pd.to_numeric, errors='coerce')
    
    # C. 自动化平稳化变形 (根据指令自动洗干净整张表的历史)
    for col in macro_cols:
        if col in df_vintage.columns:
            try: t_code = int(float(transform_dict[col]))
            except: t_code = 1
            
            if t_code == 1: pass
            elif t_code == 2: df_vintage[col] = df_vintage[col].diff()
            elif t_code == 3: df_vintage[col] = df_vintage[col].diff().diff()
            elif t_code == 4: df_vintage[col] = np.log(df_vintage[col])
            elif t_code == 5: df_vintage[col] = np.log(df_vintage[col]).diff()
            elif t_code == 6: df_vintage[col] = np.log(df_vintage[col]).diff().diff()
            elif t_code == 7:
                df_vintage[col] = (
                    df_vintage[col]
                    .pct_change(fill_method=None)
                    .diff()
                )
            else:
                raise ValueError(f"Unknown transformation code: {t_code}")
            # 将df_vintage最后一行非0的值赋值给master_panel的对应位置

            master_panel.loc[cutoff_date, col] = df_vintage[col].dropna().iloc[-1] 

master_panel


>>> Target Timeline: 1999-12-01 to 2024-12-31 (301 months)


,PAYEMS,UNRATE,INDPRO,RPI,ISRATIOx,HOUST,BUSLOANS,CPIAUCSL
mthcaldt,,,,,,,,
1999-12-31,0.001808,0.0,0.002986,0.004062,0.0,7.377759,0.015038,-0.000597
2000-01-31,0.002428,0.0,0.004044,0.001568,0.0,7.445418,-0.017497,0.001184
2000-02-29,0.002975,-0.1,0.00973,0.004926,-0.01,7.481556,0.00128,-0.000597
2000-03-31,0.00033,0.1,0.002812,0.004926,-0.01,7.48493,0.010133,0.003533
2000-04-30,0.003188,0.0,0.00275,-0.00021,0.0,7.380256,-0.000968,0.001726
...,...,...,...,...,...,...,...,...
2024-08-31,0.000718,0.2,-0.006401,0.001581,0.0,7.121252,-0.006629,0.00211
2024-09-30,0.000895,-0.1,0.00811,0.001128,-0.01,7.212294,0.003711,0.000322
2024-10-31,0.001598,-0.1,-0.002832,0.001122,0.01,7.210818,0.002598,-0.000073


In [10]:
import wrds
import numpy as np
import pandas as pd
from pandas_datareader import data as web

# 其他日频金融指标
daily = web.DataReader(
    ["VIXCLS", "DFF", "DGS10", "DTB3", "DBAA"],
    "fred", start_date, end_date)

daily["TERM_SPREAD"] = daily["DGS10"] - daily["DTB3"]
daily["CREDIT_SPREAD"] = daily["DBAA"] - daily["DGS10"]

monthly = daily[
    ["VIXCLS", "DFF", "TERM_SPREAD", "CREDIT_SPREAD"]
].resample("ME").last()

monthly["FEDFUNDS"] = monthly["DFF"].diff()
monthly
monthly = monthly.drop(columns="DFF")
monthly.index.name = 'mthcaldt'
monthly.index = pd.to_datetime(monthly.index) + pd.offsets.MonthEnd(0)  # 确保日期是月末
monthly

,VIXCLS,TERM_SPREAD,CREDIT_SPREAD,FEDFUNDS
mthcaldt,,,,
1999-12-31,24.64,1.28,1.73,NaN
2000-01-31,24.95,1.15,1.68,1.88
2000-02-29,23.37,0.80,1.91,-0.02
2000-03-31,24.11,0.31,2.30,0.32
2000-04-30,26.20,0.57,2.34,-0.11
...,...,...,...,...
2024-08-31,15.00,-1.07,1.69,0.00
2024-09-30,16.73,-0.71,1.63,-0.50
2024-10-31,23.16,-0.16,1.46,0.00


In [11]:
# download S&P 500 monthly returns from WRDS
import wrds

# Connect to WRDS
db = wrds.Connection(wrds_username='jianbinchen')

# S&P 500：CRSP月度收益
sp = db.raw_sql(f"""
    SELECT date, sprtrn
    FROM crsp.msi
    WHERE date BETWEEN '{start_date}' AND '{end_date}'
""", date_cols=["date"])
sp["date"]= pd.to_datetime(sp["date"])+ pd.offsets.MonthEnd(0)  # 确保日期是月末
sp.set_index("date", inplace=True)
sp.index.name = 'mthcaldt'
display(sp)


Loading library list...
Done


,sprtrn
mthcaldt,
1999-12-31,0.057844
2000-01-31,-0.050904
2000-02-29,-0.020108
2000-03-31,0.09672
2000-04-30,-0.030796
...,...
2024-08-31,0.022835
2024-09-30,0.020197
2024-10-31,-0.009897


In [13]:
# combine all data sources into a single DataFrame
macro_data = pd.concat([master_panel, monthly, sp], axis=1)
macro_data = macro_data.reset_index()
macro_data.to_parquet(os.path.join(output_save_path, 'macro_data.parquet'), index=False)

display(macro_data)

,mthcaldt,PAYEMS,UNRATE,INDPRO,RPI,ISRATIOx,HOUST,BUSLOANS,CPIAUCSL,VIXCLS,TERM_SPREAD,CREDIT_SPREAD,FEDFUNDS,sprtrn
0,1999-12-31,0.001808,0.0,0.002986,0.004062,0.0,7.377759,0.015038,-0.000597,24.64,1.28,1.73,NaN,0.057844
1,2000-01-31,0.002428,0.0,0.004044,0.001568,0.0,7.445418,-0.017497,0.001184,24.95,1.15,1.68,1.88,-0.050904
2,2000-02-29,0.002975,-0.1,0.00973,0.004926,-0.01,7.481556,0.00128,-0.000597,23.37,0.80,1.91,-0.02,-0.020108
3,2000-03-31,0.00033,0.1,0.002812,0.004926,-0.01,7.48493,0.010133,0.003533,24.11,0.31,2.30,0.32,0.09672
4,2000-04-30,0.003188,0.0,0.00275,-0.00021,0.0,7.380256,-0.000968,0.001726,26.20,0.57,2.34,-0.11,-0.030796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,2024-08-31,0.000718,0.2,-0.006401,0.001581,0.0,7.121252,-0.006629,0.00211,15.00,-1.07,1.69,0.00,0.022835
297,2024-09-30,0.000895,-0.1,0.00811,0.001128,-0.01,7.212294,0.003711,0.000322,16.73,-0.71,1.63,-0.50,0.020197
298,2024-10-31,0.001598,-0.1,-0.002832,0.001122,0.01,7.210818,0.002598,-0.000073,23.16,-0.16,1.46,0.00,-0.009897
299,2024-11-30,0.000075,0.0,-0.002629,0.003545,0.0,7.178545,-0.000157,0.000641,13.51,-0.21,1.44,-0.25,0.057301


In [14]:
# read Macro_data from parquet file
Macro_data = pd.read_parquet(os.path.join(output_save_path, 'macro_data.parquet'))
Macro_data

,mthcaldt,PAYEMS,UNRATE,INDPRO,RPI,ISRATIOx,HOUST,BUSLOANS,CPIAUCSL,VIXCLS,TERM_SPREAD,CREDIT_SPREAD,FEDFUNDS,sprtrn
0,1999-12-31,0.001808,0.0,0.002986,0.004062,0.00,7.377759,0.015038,-0.000597,24.64,1.28,1.73,NaN,0.057844
1,2000-01-31,0.002428,0.0,0.004044,0.001568,0.00,7.445418,-0.017497,0.001184,24.95,1.15,1.68,1.88,-0.050904
2,2000-02-29,0.002975,-0.1,0.009730,0.004926,-0.01,7.481556,0.001280,-0.000597,23.37,0.80,1.91,-0.02,-0.020108
3,2000-03-31,0.000330,0.1,0.002812,0.004926,-0.01,7.484930,0.010133,0.003533,24.11,0.31,2.30,0.32,0.09672
4,2000-04-30,0.003188,0.0,0.002750,-0.000210,0.00,7.380256,-0.000968,0.001726,26.20,0.57,2.34,-0.11,-0.030796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,2024-08-31,0.000718,0.2,-0.006401,0.001581,0.00,7.121252,-0.006629,0.002110,15.00,-1.07,1.69,0.00,0.022835
297,2024-09-30,0.000895,-0.1,0.008110,0.001128,-0.01,7.212294,0.003711,0.000322,16.73,-0.71,1.63,-0.50,0.020197
298,2024-10-31,0.001598,-0.1,-0.002832,0.001122,0.01,7.210818,0.002598,-0.000073,23.16,-0.16,1.46,0.00,-0.009897
299,2024-11-30,0.000075,0.0,-0.002629,0.003545,0.00,7.178545,-0.000157,0.000641,13.51,-0.21,1.44,-0.25,0.057301


# --------------------------------------------
# 微观因子 -- WRDS CRSP Ratios 
# --------------------------------------------

这是量化基本面研究中最正统、也最纯粹的武器库——**WRDS Financial Ratios Suite (WRDS 官方财务比率库)**。

对于你目前暂时搁置 OAP 异象数据、决定专心用基本面和宏观数据跑通 MDN 基线模型（Baseline）的战略来说，这张表简直完美。它最大的价值在于：WRDS 官方已经帮你把极其复杂的底层财报科目，计算成了**对冲基金最常用的标准化比率**，并且完美对齐了 `public_date`（无前视偏差）。

面对这近 90 个变量，你绝对不能全塞进神经网络（会面临严重的多重共线性和过拟合）。结合你预测 **$ES_{5\%}$ (极左尾崩盘风险)** 的目标，我为你将这些指标分为 **5 大核心战区**，并为你挑出 MDN 最需要的“尾部特征”。

---

### 🚨 战区一：债务与破产雷区 (Solvency & Leverage) - **最核心的左尾触发器**
对于预测极值崩盘，**“公司欠了多少钱，还不还得上”**永远是第一位的。高杠杆公司在宏观环境恶化时，最容易出现断崖式下跌。

* **`short_debt` (短期债务 / 总债务):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 极度致命。短债比例越高，资金链断裂的风险呈指数级上升。
* **`intcov_ratio` (利息保障倍数):** **【MDN 必选 ⭐⭐⭐⭐】** EBIT / 利息支出。衡量公司赚的钱够不够还利息。低于 1 的公司随时暴雷。
* **`debt_at` (总债务 / 总资产):** **【MDN 必选 ⭐⭐⭐】** 最经典的宏观杠杆率。
* **`debt_ebitda` (总债务 / EBITDA):** 华尔街信用评级最看重的指标，比 `debt_at` 更能反映真实的偿债压力。
* *其他同类冗余指标 (建议剔除其一):* `lt_debt`, `debt_capital`, `de_ratio`, `totdebt_invcap`（选 `debt_at` 即可，没必要全放）。

### 🩸 战区二：流动性与现金流 (Liquidity & Cash Flow) - **抗风险的护城河**
当市场恐慌时，利润只是会计数字，**现金才是命**。缺乏流动性的公司在遭遇做空或行业寒冬时毫无抵抗力。

* **`ocf_lct` (经营现金流 / 流动负债):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 这是最真实的短期生存指标。赚不到真金白银还要面临短期债务的公司，左尾极厚。
* **`cash_ratio` (现金比率):** **【MDN 必选 ⭐⭐⭐⭐】** 账面纯现金 / 流动负债。直接衡量“今天如果债主逼债，能不能活下来”。
* **`fcf_ocf` (自由现金流 / 经营现金流):** 反映公司的资本开支压力。
* *其他同类冗余指标:* `curr_ratio` (流动比率), `quick_ratio` (速动比率)。在预测崩盘时，它们不如 `cash_ratio` 狠，因为存货和应收账款在破产时往往一文不值。

### 🎭 战区三：盈余质量 (Accounting Quality) - **财务造假与爆雷预测**
* **`accrual` (应计项目 / 平均资产):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 盈余操纵的试金石！如果一家公司净利润很高，但全在应收账款和存货里（高 accrual），这是未来业绩大爆雷（股价单日腰斩）的最强预兆。

### 💰 战区四：估值与均值回归 (Valuation & Multiples) - **决定下跌的空间**
估值越高，泡沫破裂时的“重力加速度”越大。估值因子是决定收益率分布中心 $\mu$ 和方差 $\sigma$ 的重要锚点。

* **`bm` (账面市值比):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 最经典的价值因子。
* **`pe_exi` (扣非市盈率):** **【MDN 必选 ⭐⭐⭐⭐】** 剔除了非经常性损益的真实市盈率，比 `pe_inc` 更干净。
* **`evm` (企业价值倍数 EV/EBITDA):** 并购和私募股权最爱看的指标，不受资本结构（负债率）影响。
* *其他同类冗余指标:* `pe_op_basic`, `ptb`, `ps` (市销率), `pcf` (市现率)。选 2-3 个核心的即可。

### ⚙️ 战区五：盈利能力与效率 (Profitability & Efficiency)
* **`roe` (净资产收益率):** **【MDN 必选 ⭐⭐⭐⭐】** 基本面质量的终极体现。
* **`gpm` (毛利率) 或 `opmad` (税后营业利润率):** 衡量公司在行业内的护城河深度。毛利被压缩通常是崩盘的早期信号。
* **`at_turn` (总资产周转率):** 运营效率。
* *其他同类冗余指标:* `roa`, `roce`, `aftret_eq` 等高度相关，留 `roe` 足矣。

---

### 🗑️ 垃圾桶：绝对不要放入神经网络的变量

1.  **标识符与日期：** `gvkey`, `adate`, `qdate`, `public_date`（仅用于合并和排序，绝对不能作为 $X$ 输入网络）。
2.  **文本与描述：** `gicdesc`, `ffi48_desc` 等一切 `VARCHAR` 文本列。
3.  **绝对数值：** `mktcap` (总市值), `price` (股价)。**如果你要放，必须对它们进行横截面 Rank 转换**，否则几万亿的市值会瞬间引起梯度爆炸。

---

### 🚀 你的 MDN 基线特征包 (Baseline Feature Set) 建议

为了避免维度灾难，我建议你从这张表里**精挑细选这 15 个黄金比率**作为 $X_{Fundamentals}$：

```python
mdn_features_wrds = [
    # 破产与债务
    'short_debt', 'intcov_ratio', 'debt_at', 'debt_ebitda',
    # 流动性与造假预警
    'ocf_lct', 'cash_ratio', 'accrual', 
    # 估值泡沫
    'bm', 'pe_exi', 'evm', 'divyield',
    # 盈利与效率
    'roe', 'gpm', 'at_turn', 'rd_sale'  # rd_sale (研发占比) 对美股科技股防雷很有用
]
```

将这 15 个特征，加上你处理好的 Goyal 宏观特征（如 `TAIL`, `DEF`, `VIXCLS`），你的第一版输入层 $X_T$ 就能控制在 20 个维度左右。配合上横截面 Rank 标准化，这是一个极其清爽且具备极强经济学解释力的输入矩阵！

这些财务比率中必定存在大量的 `NaN`（比如很多科技公司没有短期债务，或者尚未盈利导致市盈率缺失）。面对这些残缺，在进行 Rank 转换之前，你打算使用全市场的截面中位数填充，还是基于它所在的行业 (`gsector` / `ffi48`) 进行同行业中位数填充？

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行
wrds_path = '/Users/jianbinchen/NonSync/GitHub/Research/DataSet/WRDS.sqlite'
# set current working directory to the location of the database file
os.chdir('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/github')

In [2]:
crsp_wrds_msfv2_query_1990=pd.read_parquet('../data/crsp_wrds_msfv2_query_1990.parquet', engine='pyarrow')
pd.concat([crsp_wrds_msfv2_query_1990.head(3), crsp_wrds_msfv2_query_1990.tail(3)], axis=0)

,permno,mthcaldt,permco,mthret,mthretx,mthprc,mthcap,mthvol,shrout,naics,siccd,mthdelflg,delreasontype,year
0,10001,1990-01-31,7953,-0.018519,-0.018519,9.9375,1.015613e+04,3.525500e+04,1022,0,4920,N,UNAV,1990
1,16555,1990-01-31,21838,-0.084507,-0.084507,32.5000,5.502575e+05,4.152000e+05,16931,0,5159,N,NACT,1990
2,60506,1990-01-31,3487,-0.105263,-0.105263,38.2500,1.334542e+06,1.397552e+06,34890,0,3711,N,NACT,1990
2131996,22561,2025-12-31,59083,-0.430882,-0.430882,3.8700,3.525570e+03,1.474857e+06,911,541618,8748,N,NACT,2025
2131997,15795,2025-12-31,55468,-0.094903,-0.094903,15.4500,1.998334e+06,3.875427e+07,129342,621111,8011,N,NACT,2025
2131998,93436,2025-12-31,53453,0.045447,0.045447,449.7200,1.495687e+09,1.618322e+09,3325819,336110,3711,N,NACT,2025


In [3]:
crsp_wrds_msfv2_query_1990=pd.read_parquet('../data/crsp_wrds_msfv2_query_1990.parquet', engine='pyarrow',columns=['permno', 'mthcaldt', 'mthret', 'mthprc', 'mthcap', 'mthvol', 'shrout', 'siccd'])

# ----------------------------------------
# 剔除金融行业 (SIC 代码 6000 至 6999) 暂时作废
# crsp_wrds_msfv2_query_1990['siccd']=crsp_wrds_msfv2_query_1990['siccd'].astype('Int64')
# crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990[(crsp_wrds_msfv2_query_1990['siccd'] < 6000) | (crsp_wrds_msfv2_query_1990['siccd'] > 6999)]
# crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990.drop(columns=['siccd'])
# ----------------------------------------

print(f"total rows: {len(crsp_wrds_msfv2_query_1990)}")
# crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990.drop_duplicates(subset=['permno', 'mthcaldt'], keep='first')
crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990.drop_duplicates()
print(f"total rows after drop duplicates: {len(crsp_wrds_msfv2_query_1990)}")
print(f"number of duplicates: {crsp_wrds_msfv2_query_1990.duplicated(subset=['permno', 'mthcaldt']).sum()}")
display(pd.concat([crsp_wrds_msfv2_query_1990.head(3), crsp_wrds_msfv2_query_1990.tail(3)], axis=0))

total rows: 2131999
total rows after drop duplicates: 2126908
number of duplicates: 0


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd
0,10001,1990-01-31,-0.018519,9.9375,1.015613e+04,3.525500e+04,1022,4920
1,16555,1990-01-31,-0.084507,32.5000,5.502575e+05,4.152000e+05,16931,5159
2,60506,1990-01-31,-0.105263,38.2500,1.334542e+06,1.397552e+06,34890,3711
2131996,22561,2025-12-31,-0.430882,3.8700,3.525570e+03,1.474857e+06,911,8748
2131997,15795,2025-12-31,-0.094903,15.4500,1.998334e+06,3.875427e+07,129342,8011
2131998,93436,2025-12-31,0.045447,449.7200,1.495687e+09,1.618322e+09,3325819,3711


In [4]:
crsp_wrds_msfv2_query_1990.isna().sum()

permno         0
mthcaldt       0
mthret      1560
mthprc       832
mthcap       832
mthvol       832
shrout         0
siccd          0
dtype: int64

In [5]:
MicroFactor = pd.read_excel('../data/MicroFactor.xlsx', sheet_name='select')
display(pd.concat([MicroFactor.head(3), MicroFactor.tail(3)], axis=0))
selected_features = MicroFactor[MicroFactor['select']==1]['nullable'].tolist()
comp_wrds_ratios = pd.read_parquet('../data/comp_wrds_ratios.parquet', engine='pyarrow', 
                                   columns=selected_features)
display(pd.concat([comp_wrds_ratios.head(3), comp_wrds_ratios.tail(3)], axis=0))
print("Total duplicates found: " + str(comp_wrds_ratios.duplicated(subset=['gvkey', 'public_date'], keep=False).sum()))

,name,nullable,type,comment,description,select,中文解释,旧select
0,0,gvkey,True,VARCHAR(7),Global Company Key,1.0,NaN,1.0
1,1,adate,True,DATE,Fiscal year end,NaN,NaN,NaN
2,2,qdate,True,DATE,Fiscal quarter end,NaN,NaN,NaN
92,92,ffi48,True,DOUBLE PRECISION,Fama and French 48 Industries,NaN,NaN,NaN
93,93,ffi49_desc,True,VARCHAR(5),Fama and French 49 Industries - Description,NaN,NaN,NaN
94,94,ffi49,True,DOUBLE PRECISION,Fama and French 49 Industries,1.0,NaN,1.0


,gvkey,public_date,bm,evm,ps,pcf,opmad,cfm,roe,gprof,debt_at,short_debt,cash_ratio,curr_ratio,at_turn,rd_sale,accrual,ffi12,ffi49
0,001003,1990-01-31,NaN,-2.836651,0.034461,-0.159702,-0.153649,-0.217813,-1.813523,0.488989,0.413638,0.984159,0.056425,0.960262,1.554107,0.000000,-0.355052,9.0,43.0
1,013343,1990-01-31,3.597226,6.485974,1.409560,2.241353,0.299322,0.630680,0.041494,0.040907,0.009276,1.000000,NaN,NaN,0.136665,0.000000,-0.044903,11.0,48.0
2,002484,1990-01-31,0.715879,NaN,0.360000,-17.721272,0.047622,NaN,0.116036,NaN,0.135434,0.689936,0.201836,1.769802,1.870810,0.000000,0.099620,9.0,43.0
2610317,011600,2026-03-31,0.387288,24.216030,2.060824,12.178577,0.060467,0.124571,0.103696,0.234632,0.194237,0.020752,1.070803,3.386238,0.730025,0.000000,-0.062229,3.0,19.0
2610318,074646,2026-03-31,0.470090,13.483320,3.817789,9.156156,0.352168,-0.094985,-0.103688,0.236191,0.371024,0.988021,0.569129,1.609304,0.394388,0.000000,-0.209899,NaN,NaN
2610319,027928,2026-03-31,0.137635,29.715752,5.963023,17.182012,0.270861,0.256796,0.225025,0.466616,0.194825,0.083494,0.478110,1.379504,0.570711,0.158392,-0.076517,6.0,36.0


Total duplicates found: 0


In [6]:
# merge crsp_wrds_msfv2_query_1990 and comp_wrds_ratios by ccm link
with sqlite3.connect(wrds_path) as db_conn:
    crsp_ccmxpf_linktable = pd.read_sql('SELECT * FROM crsp_ccmxpf_lnkhist', db_conn, parse_dates=['linkdt','linkenddt'])
crsp_ccmxpf_linktable['linkenddt'] = pd.to_datetime(crsp_ccmxpf_linktable['linkenddt']).fillna(pd.to_datetime('2099-12-31'))
# crsp_ccmxpf_linktable['lpermno'] = crsp_ccmxpf_linktable['lpermno'].astype("Int64").astype(str).replace("<NA>", pd.NA).str.strip()
crsp_ccmxpf_linktable['lpermno'] = crsp_ccmxpf_linktable['lpermno'].astype("Int64")
# crsp_ccmxpf_linktable['gvkey'] = crsp_ccmxpf_linktable['gvkey'].astype(str).str.zfill(6)
crsp_ccmxpf_linktable['gvkey'] = crsp_ccmxpf_linktable['gvkey'].astype("Int64")
# crsp_wrds_msfv2_query_1990['permno'] = crsp_wrds_msfv2_query_1990['permno'].astype("Int64").astype(str).replace("<NA>", pd.NA).str.strip()
crsp_wrds_msfv2_query_1990['permno'] = crsp_wrds_msfv2_query_1990['permno'].astype("Int64")
# comp_wrds_ratios['gvkey'] = comp_wrds_ratios['gvkey'].astype(str).str.zfill(6)
comp_wrds_ratios['gvkey'] = comp_wrds_ratios['gvkey'].astype("Int64")

ccm = crsp_ccmxpf_linktable[(crsp_ccmxpf_linktable['linktype'].isin(['LU', 'LC'])) & (crsp_ccmxpf_linktable['linkprim'].isin(['P', 'C']))]
ccm = ccm.rename(columns={'lpermno': 'permno'})

comp_ccm = pd.merge(comp_wrds_ratios, ccm[['gvkey', 'permno', 'linkdt', 'linkenddt']], on='gvkey', how='inner')
comp_ccm = comp_ccm[(comp_ccm['public_date'] >= comp_ccm['linkdt']) & 
                    (comp_ccm['public_date'] <= comp_ccm['linkenddt'])]

comp_ccm['public_date'] = comp_ccm['public_date'] + pd.offsets.MonthEnd(0) # 把日期推到月底，和CRSP日期对齐
crsp_wrds_msfv2_query_1990['mthcaldt'] = crsp_wrds_msfv2_query_1990['mthcaldt'] + pd.offsets.MonthEnd(0)# 把日期推到月底
print(f"CRSP monthly data shape: {crsp_wrds_msfv2_query_1990.shape}")
# 匹配每一行有交易数据的股票和财报数据，注意这里是 left join，保留所有的 CRSP 行，财报数据不匹配的会被填 NaN
micro_df = pd.merge(
    crsp_wrds_msfv2_query_1990,
    comp_ccm,
    left_on=['permno', 'mthcaldt'],
    right_on=['permno', 'public_date'],
    how='left'
)
micro_df = micro_df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
micro_df = micro_df[micro_df['mthcaldt'].between(pd.to_datetime('1999-12-1'), pd.to_datetime('2025-01-31'))]  
# # =========================================================
# # 核心修复：强制对齐“日历月网格” (Resampling to Continuous Grid)
# # =========================================================
# print(" -> Reindexing panel data to a continuous monthly grid...")

# # 2. 将日期设置为 Index，为 Pandas 的重采样做准备
# micro_df = micro_df.set_index('mthcaldt')

# # 3. ★ 核心魔法：按 permno 分组，按月 ('ME' 或 'M') 重新采样
# # 只要两个日期之间有断层，Pandas 就会自动插入缺失的月份，并将那一行的值设为 NaN
# micro_df = micro_df.groupby('permno').resample('ME').asfreq().drop(columns=['permno']).reset_index()
# micro_df = micro_df.drop(columns=['linkdt', 'linkenddt', 'public_date'], errors='ignore') # 删除不再需要的列
micro_df.to_parquet('../data/micro_data.parquet', index=False)
pd.concat([micro_df.head(5), micro_df.tail(5)], axis=0)

CRSP monthly data shape: (2126908, 8)


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,bm,evm,ps,pcf,opmad,cfm,roe,gprof,debt_at,short_debt,cash_ratio,curr_ratio,at_turn,rd_sale,accrual,ffi12,ffi49,linkdt,linkenddt
119,10001,1999-12-31,-0.004679,8.50000,2.082500e+04,3.248800e+04,2450,4920,12994,1999-12-31,0.648367,8.382672,0.358747,3.487122,0.056276,0.057297,0.116091,0.105303,0.431281,0.166191,0.028773,1.371188,1.227584,0.000000,-0.092283,8.0,31.0,1986-01-09,2017-08-31
120,10001,2000-01-31,-0.044118,8.12500,1.990625e+04,4.033300e+04,2450,4920,12994,2000-01-31,0.648367,8.382672,0.345460,3.357969,0.056276,0.057297,0.116091,0.105303,0.431281,0.166191,0.028773,1.371188,1.227584,0.000000,-0.092283,8.0,31.0,1986-01-09,2017-08-31
121,10001,2000-02-29,0.015385,8.25000,2.021250e+04,2.217600e+04,2450,4920,12994,2000-02-29,0.627854,8.663359,0.322297,302.541045,0.049894,0.048829,0.096569,0.099794,0.440956,0.213833,0.024975,1.282040,1.292765,0.000000,0.026443,8.0,31.0,1986-01-09,2017-08-31
122,10001,2000-03-31,-0.015288,8.00000,1.971200e+04,7.232000e+04,2464,4920,12994,2000-03-31,0.627854,8.663359,0.313421,294.208955,0.049894,0.048829,0.096569,0.099794,0.440956,0.213833,0.024975,1.282040,1.292765,0.000000,0.026443,8.0,31.0,1986-01-09,2017-08-31
123,10001,2000-04-30,0.011719,8.09375,1.994300e+04,2.634000e+04,2464,4920,12994,2000-04-30,0.627854,8.663359,0.323216,303.402985,0.049894,0.048829,0.096569,0.099794,0.440956,0.213833,0.024975,1.282040,1.292765,0.000000,0.026443,8.0,31.0,1986-01-09,2017-08-31
2126892,93436,2024-09-30,0.221942,261.63000,8.390474e+08,1.604206e+09,3207000,3711,184996,2024-09-30,0.105166,57.713252,8.802612,72.758187,0.075809,0.183071,0.217032,0.207100,0.095088,0.293684,1.017029,1.760274,0.902169,0.047001,0.010123,2.0,23.0,2010-06-29,2099-12-31
2126893,93436,2024-10-31,-0.045025,249.85000,8.020335e+08,1.901431e+09,3210060,3711,184996,2024-10-31,0.105166,57.713252,8.406271,69.482219,0.075809,0.183071,0.217032,0.207100,0.095088,0.293684,1.017029,1.760274,0.902169,0.047001,0.010123,2.0,23.0,2010-06-29,2099-12-31
2126894,93436,2024-11-30,0.381469,345.16000,1.107984e+09,2.082131e+09,3210060,3711,184996,2024-11-30,0.083346,53.762150,11.394010,76.450592,0.084756,0.183922,0.208738,0.203448,0.099842,0.274887,1.047903,1.798235,0.866390,0.044858,-0.014637,2.0,23.0,2010-06-29,2099-12-31
2126895,93436,2024-12-31,0.170008,403.84000,1.298958e+09,1.894644e+09,3216517,3711,184996,2024-12-31,0.083346,53.762150,13.368497,89.698836,0.084756,0.183922,0.208738,0.203448,0.099842,0.274887,1.047903,1.798235,0.866390,0.044858,-0.014637,2.0,23.0,2010-06-29,2099-12-31
2126896,93436,2025-01-31,0.001882,404.60000,1.301403e+09,1.502504e+09,3216517,3711,184996,2025-01-31,0.083346,53.762150,13.393655,89.867643,0.084756,0.183922,0.208738,0.203448,0.099842,0.274887,1.047903,1.798235,0.866390,0.044858,-0.014637,2.0,23.0,2010-06-29,2099-12-31


In [7]:
# 按月统计全部股票每一列缺失的占比, 只打印超过20%的月份和列
missing_by_month = micro_df.groupby('mthcaldt').apply(lambda x: x.isna().mean())
missing_by_month = missing_by_month[missing_by_month > 0.2].stack().reset_index() #stack() 将多层索引的 DataFrame 转换为长格式
missing_by_month.columns = ['mthcaldt', 'column', 'missing_ratio']
# print("Months and columns with more than 20% missing values:")
# print(missing_by_month)
missing_by_month.to_excel('../data/missing_by_month.xlsx', index=False)
# 主要是divyield缺失比较严重, 已经删除了缺失值很严重的数据

/var/folders/8b/0krd96r57jldyg4t2s79cv140000gn/T/ipykernel_21397/2301910022.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_by_month = micro_df.groupby('mthcaldt').apply(lambda x: x.isna().mean())


In [38]:
# 剔除金融行业 (SIC 代码 6000 至 6999) 暂时作废
micro_df['siccd']=micro_df['siccd'].astype('Int64')
print(f"Total rows before removing financial sector: {len(micro_df)}")
micro_df=micro_df[(micro_df['siccd'] < 6000) | (micro_df['siccd'] > 6999)]
print(f"Total rows after removing financial sector: {len(micro_df)}")
micro_df=micro_df.drop(columns=['siccd'])
print(f"Total rows after dropping siccd column: {len(micro_df)}")
print(f"Total duplicates found after merging: {micro_df.duplicated(subset=['permno', 'mthcaldt'], keep=False).sum()}")

Total rows before removing financial sector: 1305253
Total rows after removing financial sector: 1037671
Total rows after dropping siccd column: 1037671
Total duplicates found after merging: 0


In [39]:
micro_df[micro_df['public_date'].isna()].sample(20)

,permno,mthcaldt,mthret,mthprc,mthcap,gvkey,public_date,bm,evm,ps,pcf,opmad,cfm,roe,gprof,debt_at,short_debt,cash_ratio,curr_ratio,at_turn,rd_sale,accrual,ffi12,ffi49,linkdt,linkenddt
565165,29938,2017-01-31,0.011460,46.78,7908392.90,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
2095319,92589,2010-12-31,-0.082681,5.27,144376.92,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
565084,29938,2010-04-30,-0.044108,60.03,3397758.03,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
940243,71985,2011-10-31,0.186441,4.90,164542.00,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
1897237,89155,2002-12-31,-0.010911,23.38,366154.18,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
290407,15332,2016-10-31,-0.091981,15.40,966873.60,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
102361,11358,2005-08-31,-0.361702,0.24,1189.44,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
2009403,90863,2005-11-30,-0.169027,8.80,142049.60,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
2022837,91066,2006-02-28,-0.036386,39.99,2629062.57,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
1973748,90337,2009-05-31,-0.100000,0.45,32400.00,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT


In [40]:
micro_df['public_date'].isna().sum()

np.int64(32408)

In [41]:
micro_df.columns

Index(['permno', 'mthcaldt', 'mthret', 'mthprc', 'mthcap', 'gvkey',
       'public_date', 'bm', 'evm', 'ps', 'pcf', 'opmad', 'cfm', 'roe', 'gprof',
       'debt_at', 'short_debt', 'cash_ratio', 'curr_ratio', 'at_turn',
       'rd_sale', 'accrual', 'ffi12', 'ffi49', 'linkdt', 'linkenddt'],
      dtype='object')

In [10]:
# micro_df.drop(columns=[ 'public_date', 'linkdt', 'linkenddt', 'log_ret'], errors='ignore', inplace=True)
micro_df.to_parquet('../data/micro_data.parquet', index=False)

# --------------------------------
下面的是以前的旧代码, 暂时没用
# --------------------------------